[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab07_files_exceptions/lab07_student.ipynb)

# Lab 07 — Files & Exceptions

### Digital Skills and Business Decisioning Using Python

Every lab so far has lived entirely inside variables that vanish the moment the notebook's runtime stops. In this lab, your code starts talking to the outside world: saving text to a file that survives after a cell finishes running, reading data back in, and handling it gracefully when something you expected to be there (a file, a valid number) isn't.

This is a shorter lab than the last two, and it's all worked examples: run each cell, read the explanation, and see how the pieces click together. In this lab you'll learn to:

* **write** text to a file, and **append** more to it later,
* **read** a file back, either all at once or one line at a time,
* handle problems gracefully with **`try` / `except`** instead of letting your program crash,
* recognize `FileNotFoundError`, the error you'll hit constantly the moment a file isn't quite where you expect it, and
* save and reload structured data (lists, dictionaries) with the **`json`** module, so a calculation you've already done doesn't have to be redone from scratch.

### Part 1: Writing to a File

To write to a file, you call `open()` with **two** arguments: the filename, and a mode telling Python what you plan to do with it. `"w"` means write mode.

In [ ]:
with open("headlines.txt", "w") as file_object:
    file_object.write("Fed holds rates steady")

The `with` keyword should look familiar: it's the same indentation-based block structure as `if` and `for`. Here it does something specific: it opens the file, hands it to you as `file_object` for the indented block, and then **closes it automatically** the moment the block ends, even if your code hits an error partway through. Closing a file properly matters: an improperly closed file can lose data that was never actually saved to disk. You'll almost never call `.close()` yourself; `with` takes care of it.

Run the cell above, then look at the file browser on the left side of Colab (the folder icon). You'll see a new file, `headlines.txt`, sitting there. Open it: it contains exactly one line, `Fed holds rates steady`.

> **Careful: opening a file in `"w"` mode erases it first.** If `headlines.txt` already had content, running the cell above would have wiped it out before writing the new line. Only use `"w"` when you actually want a fresh, empty file.

### Writing Multiple Lines

`write()` doesn't add line breaks for you. Write twice in a row without one, and the two strings end up squished onto the same line.

In [ ]:
with open("headlines.txt", "w") as file_object:
    file_object.write("Fed holds rates steady")
    file_object.write("Tech earnings disappoint")

Open `headlines.txt` again: it now reads `Fed holds rates steadyTech earnings disappoint`, all one line. The fix is the same `\n` newline character you first saw inside f-strings back in Lab 02: include it explicitly at the end of each line you write.

In [ ]:
with open("headlines.txt", "w") as file_object:
    file_object.write("Fed holds rates steady\n")
    file_object.write("Tech earnings disappoint\n")
    file_object.write("Oil prices climb on supply fears\n")

Reopen the file in the browser: three clean, separate lines. Since this is such a common job (write every item of a list, one per line), the same accumulator-style loop from Lab 05 works just as well here:

In [ ]:
headlines = ["Fed holds rates steady", "Tech earnings disappoint", "Oil prices climb on supply fears"]

with open("headlines.txt", "w") as file_object:
    for headline in headlines:
        file_object.write(headline + "\n")

### Appending to a File

If you want to add to a file without erasing what's already there, open it in **append mode**, `"a"`, instead of write mode. Anything you write lands after the existing content; if the file doesn't exist yet, Python just creates it.

In [ ]:
with open("headlines.txt", "a") as file_object:
    file_object.write("Retailer stocks rally into the close\n")

Open `headlines.txt` once more: the three original headlines are untouched, with the new one added at the end. Mixing up `"w"` and `"a"` is one of the most common file-handling mistakes; when in doubt, open the file browser and check what actually happened.

### A Note on Colab and File Storage

Every file you write in a Colab notebook (like `headlines.txt` just now) is saved to a temporary disk attached to that notebook's runtime, not to your Google Drive. This is the same ephemeral runtime you've been reminded about since Lab 01: once the runtime disconnects or recycles, those files are gone, the same way any variable would be.

That's completely fine for practicing file handling (which is all we're doing here), but if you ever need a file to survive between sessions, mount your Google Drive first and write into that folder instead:

```python
from google.colab import drive
drive.mount('/content/drive')

# then write into, e.g., '/content/drive/MyDrive/headlines.txt'
```

You don't need to run this for the lab (it opens an authorization prompt), it's here so you recognize the pattern if you need it later.

---

### Part 2: Reading From a File

`open()` without a second argument defaults to **read mode**, `"r"`. Once you have the file open, `.read()` pulls the entire contents in as a single string.

In [ ]:
with open("headlines.txt") as file_object:
    contents = file_object.read()

print(contents)

You'll notice an extra blank line after the last headline. `.read()` returns everything in the file, including the trailing `\n` after the last line, and `print()` adds one more newline of its own on top of that. `.rstrip()` (from Lab 02) cleans it up:

In [ ]:
with open("headlines.txt") as file_object:
    contents = file_object.read()

print(contents.rstrip())

### Reading Line by Line

Often you don't want one giant string; you want to work through the file **one line at a time**, the same way you'd loop over a list. A file object is directly loop-able with `for`.

In [ ]:
with open("headlines.txt") as file_object:
    for line in file_object:
        print(line)

This time every line is followed by a blank one. Same cause as before, each line already ends with an invisible `\n`, and `print()` adds its own, so you get two. `.rstrip()` on each line fixes it:

In [ ]:
with open("headlines.txt") as file_object:
    for line in file_object:
        print(line.rstrip())

### Keeping the Lines After the with Block Ends

There's a scope-like rule here that echoes Lab 06: `file_object` only works **inside** its `with` block. Once the block ends, the file is closed and `file_object` is no longer usable. If you want to keep working with the file's lines afterward, capture them in a plain list while the file is still open, using `.readlines()`.

In [ ]:
with open("headlines.txt") as file_object:
    lines = file_object.readlines()

print(f"Read {len(lines)} lines.")
for line in lines:
    print(line.rstrip())

`readlines()` reads every line into a list of strings (each one still carrying its trailing `\n`) and hands the whole list back. Unlike `file_object`, the list `lines` is a perfectly ordinary variable: it's still there long after the `with` block has closed the file, so `print(f"Read {len(lines)} lines.")` on the line right after works fine.

### Working With a File's Contents

Once text is in memory, everything from Labs 02 through 06 applies to it. A common first step is building one clean string with the newlines stripped out, using the accumulator pattern from Lab 05:

In [ ]:
with open("headlines.txt") as file_object:
    lines = file_object.readlines()

full_text = ""
for line in lines:
    full_text += line.strip() + " "

print(full_text)

From there, `.split()` (Lab 02) turns the string into a list of words for a rough word count, and `.count()` tells you how many times a specific word or phrase shows up:

In [ ]:
words = full_text.split()
print(f"Approximate word count: {len(words)}")

mentions = full_text.lower().count("stocks")
print(f"Mentions of 'stocks': {mentions}")

`.lower()` before `.count()` makes the search case-insensitive, so `"Stocks"`, `"STOCKS"`, and `"stocks"` all count. This exact pattern, read a file, split it into words, count what you care about, is the foundation of a huge amount of real text analytics, and it works identically whether the file has 4 lines or 4 million.

---

### Part 3: File Paths

So far every filename has been a bare string like `"headlines.txt"`, with no folder in front of it. That's a **relative path**: Python looks for it relative to the notebook's current working directory, which in Colab is `/content` by default (the same place the file browser opens to). This is why the files you wrote in Part 1 show up right there at the top level of the file browser.

In [ ]:
import os

print(os.getcwd())

If your files live inside a subfolder, include it in the path, forward slashes work fine in Colab: `open("data/headlines.txt")` looks for `headlines.txt` inside a folder called `data`, relative to `/content`.

An **absolute path** spells out the *entire* location, starting from the root of the filesystem, e.g., `/content/data/headlines.txt`. You'll rarely need one in Colab since everything you create lands in `/content` anyway, but it's worth recognizing: if `open()` ever complains that a file doesn't exist and you're sure you spelled the name right, printing `os.getcwd()` (like the cell above) is the fastest way to check where Python is actually looking.

---

### Part 4: Exceptions, Handling Errors Gracefully

Whenever Python hits a problem it doesn't know how to recover from on its own (dividing by zero, opening a file that isn't there), it raises an **exception**: a special object describing what went wrong. Left alone, an exception halts the program and prints a traceback, the same red error text you've probably already seen once or twice by accident.

A `try` / `except` block lets you catch a specific exception and decide what happens instead of crashing.

In [ ]:
numbers = []

# print(sum(numbers) / len(numbers))
# ^ this line would crash with: ZeroDivisionError: division by zero

That's the `average()` function's worst-case input from Lab 06: an empty list. `len(numbers)` is `0`, and dividing by `0` is exactly the kind of thing Python refuses to do. Instead of letting that crash the whole notebook, wrap it in `try` / `except`:

In [ ]:
numbers = []

try:
    result = sum(numbers) / len(numbers)
except ZeroDivisionError:
    print("Can't average an empty list.")
else:
    print(f"Average: {result}")

Python runs the `try` block first. If it succeeds, the `else` block runs and shows the result. If it raises the exception named in `except`, Python jumps straight there instead, and the `else` block is skipped entirely. Try changing `numbers` back to something like `[10, 20, 30]` and rerun: this time the `try` block succeeds, so `else` prints the average and the `except` message never appears.

The rule of thumb: put only the line that might fail inside `try`, put whatever depends on it succeeding inside `else`, and use `except` purely for the recovery message.

### Handling a Missing File: FileNotFoundError

The single most common exception you'll hit when working with files is trying to open one that isn't there, a typo in the filename, a file that hasn't been uploaded yet, or one you deleted by mistake. Python raises a `FileNotFoundError` for exactly this.

In [ ]:
try:
    with open("reviews.txt") as file_object:
        contents = file_object.read()
except FileNotFoundError:
    print("Sorry, reviews.txt does not exist.")
else:
    print(contents)

`reviews.txt` was never created in this notebook, so the `try` block fails, Python catches the `FileNotFoundError`, and the friendly message prints instead of a traceback. This matters for a reason beyond tidiness: an uncaught exception halts the **entire** notebook cell (and anything after it that depended on it), while a caught one lets everything else keep running.

### Handling Several Files, Some Missing

This really pays off once you're processing more than one file. `headlines.txt` exists (you wrote it in Part 1), but `reviews.txt` still doesn't: a loop with `try` / `except` inside skips the missing one and keeps going, instead of stopping the whole batch.

In [ ]:
filenames = ["headlines.txt", "reviews.txt"]

for filename in filenames:
    try:
        with open(filename) as file_object:
            word_count = len(file_object.read().split())
    except FileNotFoundError:
        print(f"Skipping {filename}: not found.")
    else:
        print(f"{filename}: about {word_count} words.")

Without the `try` / `except`, the missing `reviews.txt` would have crashed the loop on its second pass, and `headlines.txt` would be the only thing you ever found out about anyway (since it runs first); anything scheduled to run *after* the crash simply wouldn't happen. With it, one missing file is just a skipped line in the output.

### Failing Silently

Sometimes you don't want to report a problem at all, just quietly move on. Python's `pass` statement means "do nothing", and it's a valid `except` body.

In [ ]:
filenames = ["headlines.txt", "reviews.txt"]

for filename in filenames:
    try:
        with open(filename) as file_object:
            word_count = len(file_object.read().split())
    except FileNotFoundError:
        pass
    else:
        print(f"{filename}: about {word_count} words.")

Same loop, but `reviews.txt` now vanishes from the output with no message at all. Use this deliberately, not out of laziness: silently swallowing every error makes it much harder to notice when something is actually wrong. A missing file that a user should know about deserves a message; one that's genuinely optional (like an optional config file) is a reasonable place for `pass`.

---

### Part 5: Saving Structured Data With json

Everything you've written to a file so far has been plain text. But most of the values you actually compute in this course are lists and dictionaries, and manually turning those back into a list or dictionary after reading them as text would be tedious and error-prone. The **`json`** module does that conversion for you: it can save (almost) any Python list or dictionary straight to a file, and load it back as the exact same data structure later.

In [ ]:
import json

stock_record = {"ticker": "AAPL", "price": 189.5, "volume": 45_000_000}

with open("stock_record.json", "w") as file_object:
    json.dump(stock_record, file_object)

`json.dump(data, file_object)` takes two arguments: the data to save, and an open file (in write mode) to save it into. Open `stock_record.json` in the file browser: it looks just like the dictionary you'd type in Python, `{"ticker": "AAPL", "price": 189.5, "volume": 45000000}`. Loading it back uses `json.load()`, the mirror image:

In [ ]:
with open("stock_record.json") as file_object:
    loaded_record = json.load(file_object)

print(loaded_record)
print(loaded_record["price"])

`loaded_record` is a genuine Python dictionary again, not a string you'd have to parse yourself, so `loaded_record["price"]` works immediately. This round-trips just as cleanly with a list: try `json.dump([1, 2, 3], file_object)` and you'll get `[1, 2, 3]` back on load, not a string that merely looks like a list.

> **`json` can only store the basics: strings, numbers, Booleans, `None`, lists, and dictionaries (nested as deeply as you like).** That covers essentially everything you've built in this course so far, which is exactly why `json` is the standard, portable way to save it, in Python and in every other major programming language.

---

### Part 6: Putting It Together, Caching an Analysis

Here's a realistic pattern that combines everything in this lab: computing statistics on a text file is a bit of work, and if `headlines.txt` never changes, recomputing the same word count every time you rerun the notebook is wasted effort. Instead, **cache** the result: try to load it from a `json` file first, and only recompute it from the original text file if the cache doesn't exist yet.

In [ ]:
def compute_stats(filename):
    """Compute word count and character count for a text file."""
    with open(filename) as file_object:
        text = file_object.read()

    words = text.split()
    return {"word_count": len(words), "character_count": len(text)}


def load_or_compute_stats(filename, cache_filename):
    """Load cached stats if they exist, otherwise compute and cache them."""
    try:
        with open(cache_filename) as file_object:
            stats = json.load(file_object)
    except FileNotFoundError:
        stats = compute_stats(filename)
        with open(cache_filename, "w") as file_object:
            json.dump(stats, file_object)
        print("Computed fresh stats and cached them.")
    else:
        print("Loaded cached stats.")

    return stats


stats = load_or_compute_stats("headlines.txt", "headline_stats.json")
print(stats)

Run the cell once: `headline_stats.json` doesn't exist yet, so `load_or_compute_stats` falls into the `except` branch, calls `compute_stats` to do the real work, and saves the result for next time. Run the exact same cell again, without changing anything: this time the `try` succeeds on the first line, `else` runs instead, and you see `"Loaded cached stats."`, with no recomputation at all.

Each function here has exactly one job, `compute_stats` only computes, `load_or_compute_stats` only decides whether to load or compute, the same single-responsibility habit from Lab 06. This combination (functions, files, `try` / `except`, and `json`) is a genuinely common shape in real analysis code: expensive work gets done once, cached to disk, and reused on every run after that.

---

### Recap: Your Files & Exceptions Toolkit

| Tool | What It Does | Example |
|---|---|---|
| `open(name, "w")` | Opens a file for writing (erases existing content) | `open("out.txt", "w")` |
| `open(name, "a")` | Opens a file for appending (keeps existing content) | `open("out.txt", "a")` |
| `open(name)` | Opens a file for reading (the default mode) | `open("out.txt")` |
| `with open(...) as f:` | Opens a file and closes it automatically afterward | `with open("out.txt") as f:` |
| `f.write(text)` | Writes a string to a file (no automatic newline) | `f.write("hello\n")` |
| `f.read()` | Reads the entire file as one string | `contents = f.read()` |
| `for line in f:` | Loops over a file one line at a time | `for line in f: ...` |
| `f.readlines()` | Reads every line into a list, usable after `with` ends | `lines = f.readlines()` |
| `try: / except X:` | Runs code, and recovers if it raises exception `X` | `except ZeroDivisionError:` |
| `try: / except: / else:` | Adds a block that runs only if `try` succeeded | `else: print(result)` |
| `FileNotFoundError` | The exception raised when a file doesn't exist | `except FileNotFoundError:` |
| `pass` | Does nothing; a valid (deliberate) `except` body | `except FileNotFoundError: pass` |
| `json.dump(data, f)` | Saves a list or dictionary to an open file | `json.dump(record, f)` |
| `json.load(f)` | Loads a list or dictionary back from an open file | `record = json.load(f)` |

The whole lab in one sentence: **`open()` with the right mode gets you in or out of a file, `try` / `except` keeps a missing file or a bad calculation from crashing everything else, and `json` is how a list or dictionary survives being saved to disk and read back.**